In [1]:
import pandas as pd

gold = pd.read_parquet("data/dev_gold/gold_stress_daily.parquet")
print(gold.shape)
print(gold[["trading_date", "asset_id", "close", "korea_3y"]].head())

(102909, 34)
  trading_date asset_id     close  korea_3y
0   2000-01-04   000270    7350.0       NaN
1   2000-01-04   000660  505463.0       NaN
2   2000-01-04   005380   20700.0       NaN
3   2000-01-04   005930    6110.0       NaN
4   2000-01-04   005935    4070.0       NaN


### Read `Gold_stress_daily` parquet to csv

In [6]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():        # 노트북이 하위 폴더에서 열렸을 때 대비
    ROOT = ROOT.parent

SRC = ROOT / "data/dev_gold/gold_stress_daily.parquet"
OUT = ROOT / "outputs/gold_stress_daily.xlsx"

df = pd.read_parquet(SRC)

# Excel은 시간대(UTC) 정보가 있는 시각을 저장하지 못해서 제거합니다.
for col in df.select_dtypes(include="datetimetz").columns:
    df[col] = df[col].dt.tz_localize(None)

OUT.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT, engine="openpyxl", datetime_format="YYYY-MM-DD", date_format="YYYY-MM-DD") as writer:
    df.to_excel(writer, index=False, sheet_name="gold")
    ws = writer.sheets["gold"]
    ws.freeze_panes = "A2"                           # 헤더 고정
    col = df.columns.get_loc("asset_id") + 1
    for row in ws.iter_rows(min_row=2, min_col=col, max_col=col):
        row[0].number_format = "@"                   # 종목코드 앞의 0 유지

print(f"저장 완료: {OUT} ({len(df):,}행, {len(df.columns)}컬럼)")

저장 완료: c:\Users\손성진\Desktop\stresslab\outputs\gold_stress_daily.xlsx (102,909행, 34컬럼)
